### This a the list of codes I used to debug the CNN ranker for giving nan Train MSE in the first epoch

# 1.Speed Update: Shift training from CPU to Apple MPS

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device)

# 2. Check for nulls

# Check for NaN
na_cols = [col for col in feat_cols if train_df[col].isna().sum() > 0]
print(f'Columns with NaN: {len(na_cols)}')
if na_cols:
    print(train_df[na_cols].isna().sum())

# Check for inf
inf_cols = [col for col in feat_cols if np.isinf(train_df[col]).sum() > 0]
print(f'Columns with inf: {len(inf_cols)}')

# Check labels
print(f'NaN in labels: {train_df["label"].isna().sum()}')

# Check if nan are produced in the input or output

model_cnn.eval()
sample_x, sample_y = next(iter(train_loader))
sample_x = sample_x.to(device)
sample_y = sample_y.to(device)

print("Input NaN:", torch.isnan(sample_x).any().item())
print("Input Inf:", torch.isinf(sample_x).any().item())
print("Label NaN:", torch.isnan(sample_y).any().item())

with torch.no_grad():
    out = model_cnn(sample_x)
    print("Output NaN:", torch.isnan(out).any().item())
    print("Output shape:", out.shape)
    print("Output sample:", out[:5])

### Check the layer in which inputs are found

model_cnn.eval()
x = sample_x  # already on mps

with torch.no_grad():
    print("Input:", torch.isnan(x).any().item())
    
    x1 = model_cnn.conv_block[0](x)  # first Conv1d
    print("After Conv1d:", torch.isnan(x1).any().item())
    
    x2 = model_cnn.conv_block[1](x1)  # BatchNorm or activation
    print("After layer 1:", torch.isnan(x2).any().item())
    
    x3 = model_cnn.conv_block[2](x2)  # next layer
    print("After layer 2:", torch.isnan(x3).any().item())

# Check the scale of your inputs
print("Input mean:", sample_x.mean().item())
print("Input std:",  sample_x.std().item())
print("Input min:",  sample_x.min().item())
print("Input max:",  sample_x.max().item())

# Check conv weight scale
conv_layer = model_cnn.conv_block[0]
print("\nConv weight mean:", conv_layer.weight.mean().item())
print("Conv weight std:",  conv_layer.weight.std().item())
print("Conv weight max:",  conv_layer.weight.abs().max().item())

# Check if mps is broken

import torch
print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("MPS built:", torch.backends.mps.is_built())

# Test if MPS itself is broken on your install
x = torch.randn(4, 64).to('mps')
lin = torch.nn.Linear(64, 32).to('mps')
out = lin(x)
print("Simple Linear on MPS NaN:", torch.isnan(out).any().item())
print("Simple Linear output:", out[0, :3])

In [ ]:
give me a full eli5 pdf detailing the difference between using cpu vs mps vs cuda for dL training. Then include the full summary of issues I faced in training the cnn so far and the fixes we did. 